# DSM S-Curve Manifold

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Noisy samples and denoising score vectors on an S-curve.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/dsm-scurve/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(42)

# The demo's default noise level: data on the curve y = sin(2x), x in [-1.5, 2.5]
sigma = 0.2


def sample_clean(n):
    t = -1.5 + 4 * torch.rand(n)
    return torch.stack([t, torch.sin(2 * t)], dim=1)


# ── Denoising score matching: regress s(x~) onto -(x~ - x) / sigma^2 ────────
net = torch.nn.Sequential(torch.nn.Linear(2, 128), torch.nn.SiLU(),
                          torch.nn.Linear(128, 128), torch.nn.SiLU(), torch.nn.Linear(128, 2))
opt = torch.optim.Adam(net.parameters(), lr=2e-3)
for step in range(1, 2001):
    x = sample_clean(512)
    noisy = x + sigma * torch.randn_like(x)
    target = -(noisy - x) / sigma ** 2
    loss = ((net(noisy) - target) ** 2).sum(1).mean() * sigma ** 2     # sigma^2 weighting keeps it O(1)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step in (1, 500, 2000):
        print(f"step {step:4d}: DSM loss {loss.item():.4f}")

# The loss levels off above 0: each target is a noisy score, so even the true score
# cannot fit it exactly. Judge the result by where the learned field points.
# ── The learned score points from off-curve points back toward the curve ────
with torch.no_grad():
    t = torch.linspace(-1.2, 2.2, 7)
    on = torch.stack([t, torch.sin(2 * t)], 1)
    off = on + torch.tensor([0.0, 0.35])                    # lift points above the curve
    s = net(off)
    print("\n point above the curve     learned score       y-component < 0 (points down)?")
    for p_, v in zip(off.tolist(), s.tolist()):
        print(f"({p_[0]:+.2f}, {p_[1]:+.2f})          ({v[0]:+.2f}, {v[1]:+.2f})        {v[1] < 0}")
    # A step along the score moves the point closer to the curve
    moved = off + sigma ** 2 * s
    dist = lambda q: (q[:, 1] - torch.sin(2 * q[:, 0])).abs().mean().item()
    print(f"\nmean vertical gap to the curve: before {dist(off):.3f}, after one step x + sigma^2 s(x): {dist(moved):.3f}")

step    1: DSM loss 2.0226


step  500: DSM loss 0.9386


step 2000: DSM loss 0.9931

 point above the curve     learned score       y-component < 0 (points down)?
(-1.20, -0.33)          (-3.58, -3.29)        True
(-0.63, -0.60)          (+2.28, -4.52)        True
(-0.07, +0.22)          (+3.81, -1.65)        True
(+0.50, +1.19)          (+4.05, -6.38)        True
(+1.07, +1.20)          (-3.81, -6.69)        True
(+1.63, +0.23)          (-3.89, -1.94)        True
(+2.20, -0.60)          (-3.79, -4.59)        True

mean vertical gap to the curve: before 0.350, after one step x + sigma^2 s(x): 0.039
